# 01 - Global Counterfactual Influence Graph Discovery

This smile-only notebook checks out a pinned repository revision and runs one standalone, unbuffered command. Dependency installation, Kaggle asset discovery, Grad-CAM++ screening, diffusion interventions, and graph freezing are all printed in the Logs tab.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time
os.environ['PYTHONUNBUFFERED'] = '1'

REPO_URL = 'https://github.com/lokissdo/cci-diff.git'
GIT_REF = 'main'
PROJECT_ROOT = Path('/kaggle/working/cci-diff')
OUTPUT_ROOT = Path('/kaggle/working/cci_graph_discovery')
MODEL_PATH = 'sd2-community/stable-diffusion-2-1'
SAMPLE_COUNT = 300
DEVICE = 'cuda'
NUM_INFERENCE_STEPS = 35
SEED = 42
MAX_SELECTED_REGIONS = 4
STOP_FLIP_RATE = 0.96

In [ ]:
def bootstrap(label, command, cwd=None):
    print(f'[{time.strftime("%H:%M:%S")}] START {label}', flush=True)
    print('+ ' + ' '.join(str(value) for value in command), flush=True)
    subprocess.run([str(value) for value in command], cwd=cwd, check=True)
    print(f'[{time.strftime("%H:%M:%S")}] DONE  {label}', flush=True)

if not (PROJECT_ROOT / '.git').is_dir():
    bootstrap('bootstrap: clone repository', ['git', 'clone', '--no-checkout', REPO_URL, PROJECT_ROOT])
bootstrap('bootstrap: fetch pinned revision', ['git', 'fetch', '--depth', '1', 'origin', GIT_REF], cwd=PROJECT_ROOT)
bootstrap('bootstrap: checkout pinned revision', ['git', 'checkout', '--force', '--detach', 'FETCH_HEAD'], cwd=PROJECT_ROOT)

command = [
    sys.executable, '-u', PROJECT_ROOT / 'scripts' / 'run_kaggle_smile.py',
    '--mode', 'discovery', '--sample_count', str(SAMPLE_COUNT),
    '--model_path', MODEL_PATH, '--device', DEVICE,
    '--num_inference_steps', str(NUM_INFERENCE_STEPS), '--seed', str(SEED),
    '--output_dir', OUTPUT_ROOT,
]
print(f'[{time.strftime("%H:%M:%S")}] START standalone smile discovery', flush=True)
print('+ ' + ' '.join(str(value) for value in command), flush=True)
subprocess.run(command, check=True)
print(f'[{time.strftime("%H:%M:%S")}] DONE  standalone smile discovery', flush=True)

In [ ]:
import pandas as pd
metrics_path = OUTPUT_ROOT / 'smile' / 'graph' / 'region_set_metrics.csv'
metrics = pd.read_csv(metrics_path)
display(metrics.sort_values(['flip_rate', 'mean_effect'], ascending=False).head(20))
print('Graph outputs:', OUTPUT_ROOT / 'smile' / 'graph', flush=True)